In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from geo import geodata
import warnings
warnings.filterwarnings('ignore')

# ============================================
# STEP 1: INSTALL REQUIRED PACKAGE (run once)
# ============================================
# !pip install geo-nuuuwan geopandas matplotlib

# ============================================
# LOAD YOUR DATA
# ============================================
print("📊 Loading birth data...")
df = pd.read_excel("2020_Birth_Final.xlsx")
print(f"✅ Loaded {len(df):,} records")

# ============================================
# GET SRI LANKAN DISTRICT BOUNDARIES
# ============================================
print("\n🗺️ Loading Sri Lankan district boundaries from geo-nuuuwan...")

# Get geospatial data for Sri Lanka districts
# 'LK' is the country code for Sri Lanka, 'district' gets district-level boundaries
districts_gdf = geodata.get_region_geodata('LK', 'district')
print(f"✅ Loaded {len(districts_gdf)} districts")

# Display the columns to see what we have
print(f"\n📋 District data columns: {districts_gdf.columns.tolist()}")

# Check the first few districts
print("\n📍 Sample districts:")
print(districts_gdf[['name', 'name_si', 'name_ta']].head(10))

# ============================================
# IDENTIFY MOOR POPULATION
# ============================================
print("\n🔍 Identifying Sri Lankan Moor population...")

def is_moor(race):
    """Check if race is Moor"""
    if pd.isna(race):
        return False
    race = str(race).strip()
    return race in ['Srilankan Moor', 'Sri Lankan Moor', 'Moor']

# Create Moor indicator columns
df['Mother_Is_Moor'] = df['Race_of_Mother'].apply(is_moor)
df['Father_Is_Moor'] = df['Race_of_Father'].apply(is_moor)

# TOTAL Moor population in Sri Lanka
total_moor_mother = df['Mother_Is_Moor'].sum()
total_moor_father = df['Father_Is_Moor'].sum()
total_records = len(df)

print(f"\n📊 TOTAL MOOR POPULATION IN SRI LANKA:")
print(f"   • Moor mothers: {total_moor_mother:,} ({total_moor_mother/total_records*100:.2f}% of all births)")
print(f"   • Moor fathers: {total_moor_father:,} ({total_moor_father/total_records*100:.2f}% of all births)")

# ============================================
# CALCULATE MOOR COUNTS BY DISTRICT
# ============================================
print("\n🧮 Calculating Moor population by district...")

# Create a mapping dictionary for district names
# The geo-nuuuwan data uses English names that should match your data
district_mapping = {}

# Extract district names from the geodata
for idx, row in districts_gdf.iterrows():
    district_name = row['name']  # English name
    # Add variations to help with matching
    district_mapping[district_name.lower()] = district_name
    district_mapping[district_name.upper()] = district_name
    # Add common variations
    if district_name == 'Kurunegala':
        district_mapping['kurunagala'] = district_name
    if district_name == 'Anuradhapura':
        district_mapping['anuradapura'] = district_name

# Function to match district names
def match_district(district_name, mapping):
    if pd.isna(district_name):
        return None
    district_str = str(district_name).strip().lower()
    return mapping.get(district_str, None)

# Calculate Moor counts for each district from geodata
district_stats = []

for idx, row in districts_gdf.iterrows():
    district_name = row['name']
    
    # Find matching records in your data
    # Try exact match first, then partial match
    district_mask = df['Registered_District'].str.contains(district_name, case=False, na=False)
    
    # If no matches, try with common variations
    if district_mask.sum() == 0:
        if district_name == 'Kurunegala':
            district_mask = df['Registered_District'].str.contains('Kurunagala', case=False, na=False)
        elif district_name == 'Anuradhapura':
            district_mask = df['Registered_District'].str.contains('Anuradapura', case=False, na=False)
    
    district_data = df[district_mask]
    total_births = len(district_data)
    
    if total_births > 0:
        moor_mothers = district_data['Mother_Is_Moor'].sum()
        moor_fathers = district_data['Father_Is_Moor'].sum()
        
        # Percentage of TOTAL Moor population
        pct_of_total_mothers = (moor_mothers / total_moor_mother * 100) if total_moor_mother > 0 else 0
        pct_of_total_fathers = (moor_fathers / total_moor_father * 100) if total_moor_father > 0 else 0
        
        # Percentage within district
        pct_in_district_mothers = (moor_mothers / total_births * 100) if total_births > 0 else 0
        
        district_stats.append({
            'district': district_name,
            'total_births': total_births,
            'moor_mothers': moor_mothers,
            'moor_fathers': moor_fathers,
            'pct_mothers_of_total': round(pct_of_total_mothers, 2),
            'pct_fathers_of_total': round(pct_of_total_fathers, 2),
            'pct_mothers_in_district': round(pct_in_district_mothers, 2)
        })
    else:
        # District exists in map but no data
        district_stats.append({
            'district': district_name,
            'total_births': 0,
            'moor_mothers': 0,
            'moor_fathers': 0,
            'pct_mothers_of_total': 0,
            'pct_fathers_of_total': 0,
            'pct_mothers_in_district': 0
        })

# Create DataFrame
stats_df = pd.DataFrame(district_stats)

# Merge with geodata
districts_gdf = districts_gdf.merge(
    stats_df[['district', 'total_births', 'moor_mothers', 'moor_fathers', 
              'pct_mothers_of_total', 'pct_fathers_of_total', 'pct_mothers_in_district']],
    left_on='name',
    right_on='district',
    how='left'
)

# ============================================
# CREATE SIDE-BY-SIDE MAPS
# ============================================
print("\n🎨 Creating publication-quality maps...")

# Set up the figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))
fig.patch.set_facecolor('white')

# Define color maps
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# Custom colormap for percentages
colors = ['#f7fcf5', '#e5f5e0', '#c7e9c0', '#a1d99b', '#74c476', '#41ab5d', '#238b45', '#006d2c', '#00441b']
moor_cmap = LinearSegmentedColormap.from_list('moor_cmap', colors, N=100)

# ============================================
# MAP 1: Based on Mother's Race
# ============================================
# Plot districts colored by % of total Moor mothers
districts_gdf.plot(
    column='pct_mothers_of_total',
    cmap=moor_cmap,
    linewidth=0.8,
    edgecolor='white',
    ax=ax1,
    legend=False,
    alpha=0.9,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

# Add district boundaries
districts_gdf.boundary.plot(ax=ax1, color='black', linewidth=0.5, alpha=0.5)

# Add labels for districts with significant Moor population
for idx, row in districts_gdf.iterrows():
    if row['pct_mothers_of_total'] > 3:  # Only label districts with >3% of total
        # Get centroid for label placement
        centroid = row.geometry.centroid
        ax1.annotate(
            row['name'],
            xy=(centroid.x, centroid.y),
            fontsize=8,
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none')
        )

# Add title and notes
ax1.set_title(f"Mother's Race: Distribution of {total_moor_mother:,} Moor Mothers\n(% of total Moor population living in each district)", 
              fontsize=14, fontweight='bold', pad=15)
ax1.set_axis_off()

# ============================================
# MAP 2: Based on Father's Race
# ============================================
# Plot districts colored by % of total Moor fathers
districts_gdf.plot(
    column='pct_fathers_of_total',
    cmap=moor_cmap,
    linewidth=0.8,
    edgecolor='white',
    ax=ax2,
    legend=False,
    alpha=0.9,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

# Add district boundaries
districts_gdf.boundary.plot(ax=ax2, color='black', linewidth=0.5, alpha=0.5)

# Add labels for districts with significant Moor population
for idx, row in districts_gdf.iterrows():
    if row['pct_fathers_of_total'] > 3:
        centroid = row.geometry.centroid
        ax2.annotate(
            row['name'],
            xy=(centroid.x, centroid.y),
            fontsize=8,
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none')
        )

ax2.set_title(f"Father's Race: Distribution of {total_moor_father:,} Moor Fathers\n(% of total Moor population living in each district)", 
              fontsize=14, fontweight='bold', pad=15)
ax2.set_axis_off()

# ============================================
# ADD COMMON ELEMENTS
# ============================================
# Add a single colorbar for both maps
sm = plt.cm.ScalarMappable(cmap=moor_cmap, norm=plt.Normalize(vmin=0, vmax=30))
sm.set_array([])
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Percentage of Total Moor Population (%)', fontsize=11)

# Add main title
plt.suptitle('Sri Lankan Moor Population Distribution by District (2020 Birth Data)', 
             fontsize=16, fontweight='bold', y=0.98)

# Add footer with methodology
footer_text = (
    f"Based on {total_records:,} birth records | "
    f"Total Moor mothers: {total_moor_mother:,} | "
    f"Total Moor fathers: {total_moor_father:,}\n"
    f"Colors show the percentage of ALL Moor individuals living in each district. "
    f"Darker colors indicate higher concentration."
)
plt.figtext(0.5, 0.02, footer_text, ha='center', fontsize=9, 
            style='italic', color='#2c3e50',
            bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.8))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.savefig('sri_lanka_moor_maps.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Saved: sri_lanka_moor_maps.png")

# ============================================
# CREATE SUPPLEMENTARY BAR CHART
# ============================================
print("\n📊 Creating supplementary bar chart...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Filter districts with data
districts_with_data = stats_df[stats_df['total_births'] > 0].sort_values('pct_mothers_of_total', ascending=False)

# Mother's Race - Top 10
if len(districts_with_data) > 0:
    top10 = districts_with_data.head(10)
    x_pos = range(len(top10))
    
    bars1 = ax1.bar(x_pos, top10['pct_mothers_of_total'].values, 
                    color='#2E86AB', alpha=0.8, edgecolor='#1A4B6E', linewidth=1)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(top10['district'].values, rotation=45, ha='right')
    ax1.set_ylabel('Percentage of Total Moor Population (%)')
    ax1.set_title(f'Top 10 Districts - Mother\'s Race\n(% of all {total_moor_mother:,} Moor mothers)', fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars1, top10['pct_mothers_of_total'].values):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# Father's Race - Top 10
top10_father = stats_df[stats_df['total_births'] > 0].sort_values('pct_fathers_of_total', ascending=False).head(10)
if len(top10_father) > 0:
    x_pos = range(len(top10_father))
    bars2 = ax2.bar(x_pos, top10_father['pct_fathers_of_total'].values, 
                    color='#A23B72', alpha=0.8, edgecolor='#6A2347', linewidth=1)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(top10_father['district'].values, rotation=45, ha='right')
    ax2.set_ylabel('Percentage of Total Moor Population (%)')
    ax2.set_title(f'Top 10 Districts - Father\'s Race\n(% of all {total_moor_father:,} Moor fathers)', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars2, top10_father['pct_fathers_of_total'].values):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Districts with Highest Concentration of Moor Population', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('moor_top_districts.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: moor_top_districts.png")

# ============================================
# EXPORT TO EXCEL
# ============================================
print(f"\n📊 Exporting to Excel...")

excel_filename = 'sri_lanka_moor_distribution.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    stats_df.to_excel(writer, sheet_name='District_Data', index=False)
    
    # Summary sheet
    summary = pd.DataFrame({
        'Metric': [
            'Total Birth Records',
            'Total Moor Mothers',
            'Total Moor Fathers',
            'Moor % of Births (Mother)',
            'Moor % of Births (Father)',
            'Districts with Moor Population',
            'Most Concentrated District (Mother)',
            'Most Concentrated District (Father)',
            'Top 3 Districts % (Mother)',
            'Top 3 Districts % (Father)'
        ],
        'Value': [
            f"{total_records:,}",
            f"{total_moor_mother:,}",
            f"{total_moor_father:,}",
            f"{total_moor_mother/total_records*100:.2f}%",
            f"{total_moor_father/total_records*100:.2f}%",
            len(districts_with_data),
            f"{districts_with_data.iloc[0]['district']} ({districts_with_data.iloc[0]['pct_mothers_of_total']:.1f}%)",
            f"{top10_father.iloc[0]['district']} ({top10_father.iloc[0]['pct_fathers_of_total']:.1f}%)",
            f"{districts_with_data.head(3)['pct_mothers_of_total'].sum():.1f}%",
            f"{top10_father.head(3)['pct_fathers_of_total'].sum():.1f}%"
        ]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"✅ Saved: {excel_filename}")

# ============================================
# PRINT SUMMARY
# ============================================
print(f"\n{'='*80}")
print(f"✅ ANALYSIS COMPLETE")
print(f"{'='*80}")
print(f"\n📊 KEY FINDINGS:")
print(f"   • Total Moor mothers: {total_moor_mother:,} ({total_moor_mother/total_records*100:.1f}% of births)")
print(f"   • Total Moor fathers: {total_moor_father:,} ({total_moor_father/total_records*100:.1f}% of births)")
print(f"   • Districts with Moor population: {len(districts_with_data)}")

if len(districts_with_data) > 0:
    print(f"\n   Top 5 Districts (Mother's Race):")
    for i in range(min(5, len(districts_with_data))):
        row = districts_with_data.iloc[i]
        print(f"      {i+1}. {row['district']}: {row['pct_mothers_of_total']:.1f}% ({row['moor_mothers']:,} mothers)")

print(f"\n📁 FILES CREATED:")
print(f"   • Maps: sri_lanka_moor_maps.png")
print(f"   • Bar chart: moor_top_districts.png")
print(f"   • Excel data: {excel_filename}")
print(f"{'='*80}")